Mount + Clone

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Clone repo
!git clone https://github.com/AvitalSkop/genai-project.git
%cd genai-project

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
fatal: destination path 'genai-project' already exists and is not an empty directory.
/content/genai-project


Imports + utils

In [ ]:
import sys
from pathlib import Path

CODE_DIR = Path("/content/genai-project/code")

if str(CODE_DIR) not in sys.path:
    sys.path.append(str(CODE_DIR))

import utils

print("Classes:", utils.CLASS_NAMES)

Classes: ['clean', 'empty', 'finished_leftovers', 'full', 'unclassified']


load prompts from utils.py

In [ ]:
prompts = utils.load_prompts()

for cls in prompts:
    print(cls, len(prompts[cls]))

clean 70
empty 70
finished_leftovers 70
full 70
unclassified 70


load model and Stable Diffusion

In [ ]:
!pip install diffusers transformers accelerate torch torchvision pillow

In [ ]:
from diffusers import StableDiffusionPipeline
import torch

device = "cuda"

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5"
).to(device)

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

defining paths

In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/genai_project"

OUTPUT_DIR = f"{BASE_DIR}/data/synthetic_clean"

for cls in utils.CLASS_NAMES:
    os.makedirs(f"{OUTPUT_DIR}/{cls}", exist_ok=True)

print("Folders ready")

Folders ready


generation definition

In [ ]:
import random

IMAGES_PER_PROMPT = 1  # כל prompt מייצר כמה תמונות

negative_prompt = """
multiple plates, more than one plate,
people, hands, utensils,
blurry, distorted, unrealistic,
extra objects, cluttered background
"""

generate

In [ ]:
from tqdm import tqdm
import os

for cls in utils.CLASS_NAMES:
    print(f"\nGenerating {cls}...")

    save_dir = f"{OUTPUT_DIR}/{cls}"

    # כמה כבר קיימות
    existing_files = os.listdir(save_dir)
    start_idx = len(existing_files)

    print(f"Continuing from image {start_idx}")

    class_prompts = prompts[cls]

    # כמה prompts כבר נוצלו
    done = start_idx // IMAGES_PER_PROMPT

    # ממשיכים רק מהאמצע
    remaining_prompts = class_prompts[done:]

    idx = start_idx

    for p in tqdm(remaining_prompts):
        for _ in range(IMAGES_PER_PROMPT):

            image = pipe(
                p,
                negative_prompt=negative_prompt,
                height=512,
                width=512,
                num_inference_steps=40,
                guidance_scale=8.5
            ).images[0]

            filename = f"{cls}_{idx}.png"
            image.save(f"{save_dir}/{filename}")

            idx += 1

print("\n✅ Generation complete!")


Generating clean...
Continuing from image 13


  0%|          | 0/57 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

  2%|▏         | 1/57 [00:21<19:55, 21.34s/it]

  0%|          | 0/40 [00:00<?, ?it/s]

...

  0%|          | 0/40 [00:00<?, ?it/s]

  7%|▋         | 5/70 [01:54<24:55, 23.00s/it]

  0%|          | 0/40 [00:00<?, ?it/s]

  9%|▊         | 6/70 [02:17<24:30, 22.98s/it]

  0%|          | 0/40 [00:00<?, ?it/s]

In [ ]:
for cls in utils.CLASS_NAMES:
    path = f"{OUTPUT_DIR}/{cls}"
    print(cls, len(os.listdir(path)))